# 装饰器 (Decorators) 实战指南

> 本 Notebook 展示了从基础到高阶的装饰器用法，包括带参数的重试逻辑、带状态的类装饰器以及多层链式调用。

## 1. 带参数的智能重试装饰器
演示如何通过三层嵌套函数实现可配置的重试机制。

In [ ]:
import time
import random
from functools import wraps

def retry(max_attempts=3, delay=1.0):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            attempts = 0
            while attempts < max_attempts:
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    attempts += 1
                    if attempts == max_attempts:
                        raise e
                    sleep_time = delay * (2 ** (attempts - 1)) + random.uniform(0, 0.1)
                    print(f"Retrying {func.__name__} ({attempts}/{max_attempts}) in {sleep_time:.2f}s...")
                    time.sleep(sleep_time)
        return wrapper
    return decorator

@retry(max_attempts=2, delay=0.5)
def fetch_data():
    if random.random() < 0.8:
        raise ConnectionError("Server busy")
    return {"data": 123}

try:
    print(fetch_data())
except Exception as e:
    print(f"Final failure: {e}")

## 2. 类装饰器 (带状态计数)
展示如何利用类的属性存储装饰器状态。

In [ ]:
class CallCounter:
    def __init__(self, func):
        wraps(func)(self)
        self.func = func
        self.count = 0

    def __call__(self, *args, **kwargs):
        self.count += 1
        print(f"{self.func.__name__} has been called {self.count} times")
        return self.func(*args, **kwargs)

@CallCounter
def say_hello(name):
    return f"Hello {name}"

say_hello("Alice")
say_hello("Bob")

## 3. 性能基准测试
观察装饰器带来的微秒级开销。

In [ ]:
import timeit

def noop(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

def raw(x): return x

@noop
def decorated(x): return x

t1 = timeit.timeit("raw(1)", globals=globals(), number=1000000)
t2 = timeit.timeit("decorated(1)", globals=globals(), number=1000000)
print(f"Raw: {t1:.4f}s, Decorated: {t2:.4f}s")